In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from collections import Counter

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

# =========================================================
# LOAD DATA
# =========================================================

train = pd.read_csv(
    '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
)

test = pd.read_csv(
    '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
)

# =========================================================
# CLEAN TEXT
# =========================================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-z0-9 ]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# =========================================================
# CREATE TRAIN DATA
# =========================================================

rows = []

options = ['A', 'B', 'C', 'D', 'E']

for _, row in train.iterrows():

    prompt = clean_text(row['prompt'])

    correct = row['answer']

    for opt in options:

        option_text = clean_text(row[opt])

        text = (
            "question " + prompt +
            " answer " + option_text
        )

        label = 1 if opt == correct else 0

        rows.append({

            'text': text,

            'label': label
        })

new_train = pd.DataFrame(rows)

print(new_train.head())

# =========================================================
# SPLIT
# =========================================================

X_train, X_valid, y_train, y_valid = train_test_split(

    new_train['text'],

    new_train['label'],

    test_size=0.2,

    random_state=42,

    stratify=new_train['label']
)

# =========================================================
# BUILD VOCAB
# =========================================================

MAX_WORDS = 30000

counter = Counter()

for text in X_train:

    counter.update(text.split())

most_common = counter.most_common(MAX_WORDS - 2)

word2idx = {

    "<PAD>": 0,

    "<UNK>": 1
}

for idx, (word, _) in enumerate(most_common, start=2):

    word2idx[word] = idx

vocab_size = len(word2idx)

print("VOCAB SIZE:", vocab_size)

# =========================================================
# TEXT TO SEQUENCE
# =========================================================

MAX_LEN = 128

def encode_text(text):

    tokens = text.split()

    seq = []

    for token in tokens:

        seq.append(

            word2idx.get(token, 1)
        )

    seq = seq[:MAX_LEN]

    while len(seq) < MAX_LEN:

        seq.append(0)

    return seq

X_train_seq = np.array([

    encode_text(text)

    for text in X_train
])

X_valid_seq = np.array([

    encode_text(text)

    for text in X_valid
])

# =========================================================
# DATASET
# =========================================================

class MCQDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(

            X,

            dtype=torch.long
        )

        self.y = torch.tensor(

            y.values,

            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

# =========================================================
# DATALOADER
# =========================================================

train_dataset = MCQDataset(

    X_train_seq,

    y_train
)

valid_dataset = MCQDataset(

    X_valid_seq,

    y_valid
)

train_loader = DataLoader(

    train_dataset,

    batch_size=64,

    shuffle=True
)

valid_loader = DataLoader(

    valid_dataset,

    batch_size=64
)

# =========================================================
# ATTENTION
# =========================================================

class Attention(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.attention = nn.Linear(

            hidden_dim * 2,

            1
        )

    def forward(self, x):

        weights = torch.softmax(

            self.attention(x),

            dim=1
        )

        output = torch.sum(

            x * weights,

            dim=1
        )

        return output

# =========================================================
# STRONG MODEL
# =========================================================

class StrongMCQModel(nn.Module):

    def __init__(self):

        super().__init__()

        # EMBEDDING

        self.embedding = nn.Embedding(

            vocab_size,

            256,

            padding_idx=0
        )

        # CNN

        self.conv1 = nn.Conv1d(

            256,

            128,

            kernel_size=3,

            padding=1
        )

        # BiLSTM

        self.lstm = nn.LSTM(

            input_size=128,

            hidden_size=128,

            batch_first=True,

            bidirectional=True,

            dropout=0.3
        )

        # ATTENTION

        self.attention = Attention(128)

        # DROPOUT

        self.dropout = nn.Dropout(0.3)

        # FC

        self.fc1 = nn.Linear(

            256,

            128
        )

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(

            128,

            1
        )

    def forward(self, x):

        # EMBEDDING

        x = self.embedding(x)

        # CNN

        x = x.permute(0, 2, 1)

        x = self.conv1(x)

        x = torch.relu(x)

        x = x.permute(0, 2, 1)

        # LSTM

        x, _ = self.lstm(x)

        # ATTENTION

        x = self.attention(x)

        # DROPOUT

        x = self.dropout(x)

        # FC

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x.squeeze()

# =========================================================
# MODEL
# =========================================================

model = StrongMCQModel().to(device)

print(model)

# =========================================================
# LOSS
# =========================================================

criterion = nn.BCEWithLogitsLoss()

# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=0.001,

    weight_decay=1e-5
)

# =========================================================
# SCHEDULER
# =========================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode='max',

    patience=1,

    factor=0.5
)

# =========================================================
# TRAINING
# =========================================================

EPOCHS = 15

best_acc = 0

for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================

    model.train()

    train_correct = 0

    train_total = 0

    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)

        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(

            outputs,

            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            1.0
        )

        optimizer.step()

        train_loss += loss.item()

        preds = torch.sigmoid(outputs)

        preds = (preds > 0.5).float()

        train_correct += (

            preds == y_batch
        ).sum().item()

        train_total += y_batch.size(0)

    train_acc = train_correct / train_total

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_correct = 0

    val_total = 0

    val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in valid_loader:

            X_batch = X_batch.to(device)

            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(

                outputs,

                y_batch
            )

            val_loss += loss.item()

            preds = torch.sigmoid(outputs)

            preds = (preds > 0.5).float()

            val_correct += (

                preds == y_batch
            ).sum().item()

            val_total += y_batch.size(0)

    val_acc = val_correct / val_total

    scheduler.step(val_acc)

    print(

        f"Epoch {epoch+1}"

        f" | Train Loss: {train_loss:.4f}"

        f" | Train Acc: {train_acc:.4f}"

        f" | Val Loss: {val_loss:.4f}"

        f" | Val Acc: {val_acc:.4f}"
    )

# =========================================================
# CREATE TEST DATA
# =========================================================

test_rows = []

for _, row in test.iterrows():

    prompt = clean_text(row['prompt'])

    for opt in options:

        option_text = clean_text(row[opt])

        text = (

            "question " + prompt +

            " answer " + option_text
        )

        test_rows.append({

            'id': row['id'],

            'text': text,

            'option': opt
        })

new_test = pd.DataFrame(test_rows)

# =========================================================
# TEST ENCODE
# =========================================================

X_test_seq = np.array([

    encode_text(text)

    for text in new_test['text']
])

X_test_tensor = torch.tensor(

    X_test_seq,

    dtype=torch.long
)

test_loader = DataLoader(

    X_test_tensor,

    batch_size=64
)

# =========================================================
# PREDICT
# =========================================================

model.eval()

all_probs = []

with torch.no_grad():

    for X_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.sigmoid(outputs)

        all_probs.extend(

            probs.cpu().numpy()
        )

new_test['prob'] = all_probs

# =========================================================
# TOP-3 PREDICTIONS
# =========================================================

final_predictions = []

for qid in new_test['id'].unique():

    temp = new_test[

        new_test['id'] == qid
    ]

    temp = temp.sort_values(

        by='prob',

        ascending=False
    )

    top3 = temp['option'].values[:3]

    prediction = ' '.join(top3)

    final_predictions.append({

        'id': qid,

        'prediction': prediction
    })

submission = pd.DataFrame(final_predictions)

print(submission.head())

# =========================================================
# SAVE
# =========================================================

submission.to_csv(

    'submission.csv',

    index=False)

DEVICE: cuda
                                                text  label
0  question pick the best possible answer what is...      0
1  question pick the best possible answer what is...      1
2  question pick the best possible answer what is...      0
3  question pick the best possible answer what is...      0
4  question pick the best possible answer what is...      0
VOCAB SIZE: 2975


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


StrongMCQModel(
  (embedding): Embedding(2975, 256, padding_idx=0)
  (conv1): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (lstm): LSTM(128, 128, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=256, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)
Epoch 1 | Train Loss: 64.8995 | Train Acc: 0.7909 | Val Loss: 16.0844 | Val Acc: 0.8000
Epoch 2 | Train Loss: 57.9180 | Train Acc: 0.8143 | Val Loss: 10.7387 | Val Acc: 0.8790
Epoch 3 | Train Loss: 30.4350 | Train Acc: 0.9115 | Val Loss: 6.6631 | Val Acc: 0.9250
Epoch 4 | Train Loss: 16.7116 | Train Acc: 0.9514 | Val Loss: 3.6556 | Val Acc: 0.9570
Epoch 5 | Train Loss: 10.6766 | Train Acc: 0.9645 | Val Loss: 2.7334 | Val Acc: 0.9610
Epoch 6 | Train Loss: 7.5526 | Train Acc: 0.9730 | Val Loss: 1.94